# Unión de datasets

## Unión de datasets de imágenes de perfiles completos y de fragmentos centrados

Se unifican los datasets de imágenes de perfiles completos y de fragmentos centrados, ambos en imágenes cuadradas.
Se genera un nuevo dataset con la metadata correspondiente y copiando las imágenes a la carpeta de destino.
El nuevo archivo de metadata se guarda en la carpeta de destino y contiene todas las columnas del dataset de fragmentos centrados, completando con valores derivados las columnas que no existían en el dataset de perfiles completos.

In [1]:
import pandas as pd
from vasijas.config import load_config
from vasijas.metadata import (
    cargar_metadata,
    completar_metadata_completos,
    get_columnas_adicionales_metadata,
    guardar_metadata,
)
from vasijas.paths import ProjectPaths, ruta_relativa
from vasijas.archivos import copiar_estructura_archivos

In [2]:
CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)

In [3]:
# Origen y destino de datos
DATA_ORIGEN_COMPLETOS = PATHS.DATA_CUADRADOS
DATA_ORIGEN_FRACCIONADOS = PATHS.DATA_CENTRADOS
DATA_DESTINO = PATHS.DATA_UNIFICADO

## Origen y destino de datos

In [4]:
print(
    f"Dataset de imágenes de perfiles completos: {ruta_relativa(DATA_ORIGEN_COMPLETOS)}"
)
print(
    f"Dataset de imágenes de perfiles fraccionados: {ruta_relativa(DATA_ORIGEN_FRACCIONADOS)}"
)
print(f"Dataset destino: {ruta_relativa(DATA_DESTINO)}")

Dataset de imágenes de perfiles completos: data/05.cuadrados
Dataset de imágenes de perfiles fraccionados: data/08.centrados
Dataset destino: data/11.unificado


## Unificación de formato de metadata


### Metadata de perfiles completos

In [5]:
df_metadata_completos = cargar_metadata(DATA_ORIGEN_COMPLETOS)
display(df_metadata_completos.head())

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero,Almedinilla
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero,Almedinilla
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero,Almedinilla
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero,Almedinilla
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero,Almedinilla


### Metadata de fracciones de perfiles

In [6]:
df_metadata_fracciones = cargar_metadata(DATA_ORIGEN_FRACCIONADOS)
display(df_metadata_fracciones.head())

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion,id_perfil_completo,borde_origen,porcentaje_avance
0,al_13b_tope_10_centrado,No,IV,12,B,6,4,1.0,150,33,800,ibero,Almedinilla,al_13b,tope,10.0
1,al_13b_tope_20_centrado,No,IV,12,B,6,4,1.0,150,33,800,ibero,Almedinilla,al_13b,tope,20.0
2,al_13b_tope_30_centrado,No,IV,12,B,6,4,1.0,150,33,800,ibero,Almedinilla,al_13b,tope,30.0
3,al_13b_tope_40_centrado,No,IV,12,B,6,4,1.0,150,33,800,ibero,Almedinilla,al_13b,tope,40.0
4,al_13b_tope_50_centrado,No,IV,12,B,6,4,1.0,150,33,800,ibero,Almedinilla,al_13b,tope,50.0


### Columnas que están en el metadata de las fracciones que no están en el metadata de los perfiles completos

In [7]:
columnas_adicionales_metadata = get_columnas_adicionales_metadata(
    df_metadata_completos,
    df_metadata_fracciones,
)

print(
    "Columnas adicionales en el metadata de fracciones que no están en el metadata de perfiles completos:"
)
for columna in columnas_adicionales_metadata:
    print(f"- {columna}")

Columnas adicionales en el metadata de fracciones que no están en el metadata de perfiles completos:
- id_perfil_completo
- borde_origen
- porcentaje_avance


In [8]:
df_metadata_completos_completo = completar_metadata_completos(
    df_metadata_completos=df_metadata_completos,
    df_metadata_referencia=df_metadata_fracciones,
    columnas_derivadas=columnas_adicionales_metadata,
)

df_metadata_completos_completo.head()

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion,id_perfil_completo,borde_origen,porcentaje_avance
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero,Almedinilla,al_03a,completo,100.0
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero,Almedinilla,al_03b,completo,100.0
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero,Almedinilla,al_03c,completo,100.0
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero,Almedinilla,al_03d,completo,100.0
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero,Almedinilla,al_03e,completo,100.0


In [9]:
assert list(df_metadata_completos_completo.columns) == list(
    df_metadata_fracciones.columns
)

df_metadata_unificados = pd.concat(
    [df_metadata_completos_completo, df_metadata_fracciones],
    ignore_index=True,
)

assert len(df_metadata_unificados) == len(df_metadata_completos_completo) + len(
    df_metadata_fracciones
)

df_metadata_unificados

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion,id_perfil_completo,borde_origen,porcentaje_avance
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero,Almedinilla,al_03a,completo,100.0
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero,Almedinilla,al_03b,completo,100.0
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero,Almedinilla,al_03c,completo,100.0
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero,Almedinilla,al_03d,completo,100.0
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero,Almedinilla,al_03e,completo,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20695,tu_96_8_tope_50_centrado,No,II,1,B,2,1,11.0,188,95,40,ibero,Tútugi,tu_96_8,tope,50.0
20696,tu_96_8_tope_60_centrado,No,II,1,B,2,1,11.0,188,95,40,ibero,Tútugi,tu_96_8,tope,60.0
20697,tu_96_8_tope_70_centrado,No,II,1,B,2,1,11.0,188,95,40,ibero,Tútugi,tu_96_8,tope,70.0
20698,tu_96_8_tope_80_centrado,No,II,1,B,2,1,11.0,188,95,40,ibero,Tútugi,tu_96_8,tope,80.0


## Unión de imágenes PNG


In [10]:
cant_archivos_completos_copiados = copiar_estructura_archivos(
    DATA_ORIGEN_COMPLETOS, DATA_DESTINO
)

cant_archivos_fraccionados_copiados = copiar_estructura_archivos(
    DATA_ORIGEN_FRACCIONADOS, DATA_DESTINO
)

total_archivos_copiados = (
        cant_archivos_fraccionados_copiados + cant_archivos_completos_copiados
)

In [11]:
print(f"Se copiaron un total de {total_archivos_copiados} archivos.")
print(
    f"De los cuales {cant_archivos_fraccionados_copiados} pertenecen a fracciones de perfiles y {cant_archivos_completos_copiados} a perfiles completos."
)

Se copiaron un total de 20702 archivos.
De los cuales 19459 pertenecen a fracciones de perfiles y 1243 a perfiles completos.


In [12]:
path_metadata_unificado = guardar_metadata(df_metadata_unificados, DATA_DESTINO)
print(
    f"El nuevo archivo metadata ha sido guardado en: {ruta_relativa(path_metadata_unificado)}"
)

El nuevo archivo metadata ha sido guardado en: data/11.unificado/metadata.csv
